<h1>Uni2h Analyze <h1>

In [1]:
# ============================================================
#  UNI2H TenCrop — Feature File Validator & Analyzer
#  ── Single notebook cell ──────────────────────────────────
#
#  What this cell does
#  -------------------
#  1. Recursively scans output_root for every .pt file.
#  2. Validates each file (loadable? tensor? NaN/Inf-free?).
#  3. Computes descriptive statistics for every valid tensor.
#  4. Prints a structured report with:
#       A. High-level counts (total / valid / invalid)
#       B. Details of any invalid files, grouped by error type
#       C. Tensor shape distribution
#       D. Aggregate statistics across all valid files
#       E. Per-slide folder summary
# ============================================================

import os
import time
import torch
import numpy as np
from collections import defaultdict
from pathlib import Path


# ─────────────────────────────────────────────────────────────
#  CONFIGURATION  (edit these two lines as needed)
# ─────────────────────────────────────────────────────────────

output_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\UNI2H_tencrop"

# Print one status line per file while scanning?
VERBOSE = False


# ─────────────────────────────────────────────────────────────
#  SECTION 1 — Single-file validator
# ─────────────────────────────────────────────────────────────

def analyze_pt_file(file_path: str) -> dict:
    """
    Load and validate one .pt feature file.

    Checks performed
    ----------------
    1. File loads without error (not corrupted / truncated).
    2. Stored object is a torch.Tensor (not a dict, list, etc.).
    3. Tensor can be safely cast to float32 numpy array
       (guards against exotic dtypes like bfloat16).
    4. No NaN or Inf values present.

    Statistics returned for valid files
    ------------------------------------
    shape, dtype, num_elements,
    mean, std, min, max, median, l2_norm, zero_fraction.

    Parameters
    ----------
    file_path : str  – Absolute path to the .pt file.

    Returns
    -------
    dict – always contains 'valid' (bool) and 'file' (str).
    """

    result = {
        "valid":          False,
        "file":           file_path,
        "error":          None,
        # ── tensor metadata ──────────────────────────────────
        "shape":          None,
        "dtype":          None,
        "num_elements":   None,
        # ── numerical health ─────────────────────────────────
        "has_nan":        None,
        "has_inf":        None,
        # ── descriptive statistics ───────────────────────────
        "mean":           None,
        "std":            None,
        "min":            None,
        "max":            None,
        "median":         None,
        "l2_norm":        None,
        "zero_fraction":  None,
    }

    # ── 1. Load the file ──────────────────────────────────────
    try:
        # weights_only=True is the safer, modern default (PyTorch >= 2.0).
        # Falls back for older installations.
        data = torch.load(file_path, map_location="cpu", weights_only=True)
    except TypeError:
        try:
            data = torch.load(file_path, map_location="cpu")
        except Exception as e:
            result["error"] = f"Load failed: {e}"
            return result
    except Exception as e:
        result["error"] = f"Load failed: {e}"
        return result

    # ── 2. Confirm the object is a Tensor ─────────────────────
    if not isinstance(data, torch.Tensor):
        result["error"] = (
            f"Expected torch.Tensor but got {type(data).__name__}. "
            "File may store a state-dict or other composite object."
        )
        return result

    result["shape"]        = tuple(data.shape)
    result["dtype"]        = str(data.dtype)
    result["num_elements"] = int(data.numel())

    # ── 3. Convert to float32 numpy ───────────────────────────
    try:
        np_data = data.to(torch.float32).cpu().numpy()
    except Exception as e:
        result["error"] = f"Numpy conversion failed: {e}"
        return result

    # ── 4. Check for NaN / Inf ────────────────────────────────
    has_nan = bool(np.isnan(np_data).any())
    has_inf = bool(np.isinf(np_data).any())
    result["has_nan"] = has_nan
    result["has_inf"] = has_inf

    if has_nan or has_inf:
        flags = []
        if has_nan: flags.append("NaN")
        if has_inf: flags.append("Inf")
        result["error"] = f"Tensor contains invalid values: {', '.join(flags)}"
        return result

    # ── 5. Descriptive statistics ─────────────────────────────
    flat = np_data.ravel()

    result["mean"]         = float(np.mean(flat))
    result["std"]          = float(np.std(flat))
    result["min"]          = float(np.min(flat))
    result["max"]          = float(np.max(flat))
    result["median"]       = float(np.median(flat))
    result["l2_norm"]      = float(np.linalg.norm(flat))
    result["zero_fraction"]= float(np.mean(flat == 0.0))

    result["valid"] = True
    return result


# ─────────────────────────────────────────────────────────────
#  SECTION 2 — Directory scanner
# ─────────────────────────────────────────────────────────────

def scan_directory(root: str, verbose: bool = False) -> list:
    """
    Recursively find every .pt file under *root* and validate each one.

    Parameters
    ----------
    root    : str  – Root directory to scan.
    verbose : bool – If True, print one status line per file.

    Returns
    -------
    list of result dicts (one per .pt file found).
    """
    root_path = Path(root)

    if not root_path.exists():
        raise FileNotFoundError(f"output_root does not exist:\n  {root}")
    if not root_path.is_dir():
        raise NotADirectoryError(f"output_root is not a directory:\n  {root}")

    pt_files = sorted(root_path.rglob("*.pt"))
    total    = len(pt_files)

    if total == 0:
        print("⚠  No .pt files found under:", root)
        return []

    print(f"Found {total:,} .pt file(s) — scanning …\n")
    t0 = time.time()

    all_results = []
    for idx, pt_path in enumerate(pt_files, start=1):
        result = analyze_pt_file(str(pt_path))
        all_results.append(result)

        if verbose:
            tag = "✓" if result["valid"] else "✗"
            rel = pt_path.relative_to(root_path)
            print(f"  [{idx:>{len(str(total))}}/{total}] {tag}  {rel}")

    elapsed = time.time() - t0
    print(f"Scan complete — {elapsed:.1f}s elapsed.")
    return all_results


# ─────────────────────────────────────────────────────────────
#  SECTION 3 — Report printer
# ─────────────────────────────────────────────────────────────

def print_report(results: list) -> None:
    """
    Print a structured, human-readable report for the scan results.

    Report sections
    ---------------
    A. High-level counts  (total / valid / invalid)
    B. Invalid file details, grouped by error message
    C. Tensor shape distribution (with mini bar-chart)
    D. Aggregate statistics across all valid files
    E. Per-slide / per-folder breakdown
    """
    if not results:
        print("No results to report.")
        return

    SEP = "=" * 72
    sep = "-" * 72

    valid_results   = [r for r in results if r["valid"]]
    invalid_results = [r for r in results if not r["valid"]]
    total           = len(results)
    n_valid         = len(valid_results)
    n_invalid       = len(invalid_results)

    root_path = Path(output_root)

    # ── A. High-level counts ──────────────────────────────────
    print(f"\n{SEP}")
    print("  UNI2H TenCrop — Feature File Report")
    print(SEP)
    print(f"  Root directory : {output_root}")
    print(f"  Total .pt files: {total:,}")
    print(f"  ✓  Valid       : {n_valid:,}  ({100*n_valid/max(total,1):.1f} %)")
    print(f"  ✗  Invalid     : {n_invalid:,}  ({100*n_invalid/max(total,1):.1f} %)")
    print(sep)

    # ── B. Invalid file details ───────────────────────────────
    if invalid_results:
        print(f"\n  INVALID FILES ({n_invalid})")
        print(sep)

        by_error = defaultdict(list)
        for r in invalid_results:
            by_error[r["error"] or "Unknown error"].append(r["file"])

        for error_msg, files in sorted(by_error.items()):
            print(f"\n  Error type : {error_msg}")
            print(f"  Affected   : {len(files)} file(s)")
            for f in files:
                try:
                    rel = Path(f).relative_to(root_path)
                except ValueError:
                    rel = Path(f)
                print(f"    → {rel}")
        print(sep)
    else:
        print("\n  ✓  All files passed validation — no errors detected.")
        print(sep)

    # ── C. Shape distribution ─────────────────────────────────
    print(f"\n  TENSOR SHAPE DISTRIBUTION  (valid files only)")
    print(sep)

    shape_counts = defaultdict(int)
    for r in valid_results:
        shape_counts[r["shape"]] += 1

    if shape_counts:
        max_count = max(shape_counts.values())
        bar_width = 30
        for shape, count in sorted(shape_counts.items(), key=lambda x: -x[1]):
            bar_len = int(count / max_count * bar_width)
            bar     = "█" * bar_len
            pct     = 100 * count / n_valid
            print(f"  {str(shape):<28} {count:>6} files  ({pct:>5.1f}%)  {bar}")
    else:
        print("  (no valid files)")
    print(sep)

    if n_valid == 0:
        print("\n  No valid files — aggregate statistics skipped.")
        print(SEP + "\n")
        return

    # ── D. Aggregate statistics ───────────────────────────────
    print(f"\n  AGGREGATE STATISTICS  (across {n_valid:,} valid files)")
    print(sep)
    print(f"  {'Metric':<22}  {'Mean':>11}  {'Std':>11}  {'Min':>11}  {'Max':>11}")
    print(f"  {'-'*22}  {'-'*11}  {'-'*11}  {'-'*11}  {'-'*11}")

    def stat_row(label, arr):
        print(f"  {label:<22}  "
              f"{np.mean(arr):>11.5f}  "
              f"{np.std(arr):>11.5f}  "
              f"{np.min(arr):>11.5f}  "
              f"{np.max(arr):>11.5f}")

    stat_row("Feature mean",   [r["mean"]   for r in valid_results])
    stat_row("Feature std",    [r["std"]    for r in valid_results])
    stat_row("Feature min",    [r["min"]    for r in valid_results])
    stat_row("Feature max",    [r["max"]    for r in valid_results])
    stat_row("Feature median", [r["median"] for r in valid_results])
    stat_row("L2 norm",        [r["l2_norm"] for r in valid_results])

    avg_zero_pct  = np.mean([r["zero_fraction"] for r in valid_results]) * 100
    total_elements = sum(r["num_elements"] for r in valid_results)

    print(f"\n  Avg zero-value fraction : {avg_zero_pct:.3f}%")
    print(f"  Total stored elements   : {total_elements:,}")
    print(sep)

    # ── E. Per-folder / per-slide breakdown ───────────────────
    print(f"\n  PER-SLIDE FOLDER SUMMARY")
    print(sep)
    print(f"  {'Folder':<32} {'Files':>5}  {'Avg Mean':>10}  "
          f"{'Avg Std':>10}  {'Shape(s)'}")
    print(f"  {'-'*32}  {'-'*5}  {'-'*10}  {'-'*10}  {'-'*16}")

    by_folder = defaultdict(list)
    for r in valid_results:
        folder = Path(r["file"]).parent.name
        by_folder[folder].append(r)

    for folder in sorted(by_folder.keys()):
        entries  = by_folder[folder]
        n        = len(entries)
        avg_mean = np.mean([r["mean"] for r in entries])
        avg_std  = np.mean([r["std"]  for r in entries])
        shapes   = ", ".join(str(s) for s in sorted(set(r["shape"] for r in entries)))
        print(f"  {folder:<32} {n:>5}  {avg_mean:>10.5f}  {avg_std:>10.5f}  {shapes}")

    print(SEP)
    print("  Report complete.")
    print(SEP + "\n")


# ─────────────────────────────────────────────────────────────
#  SECTION 4 — Run
# ─────────────────────────────────────────────────────────────

results = scan_directory(output_root, verbose=VERBOSE)
print_report(results)

Found 1,377,933 .pt file(s) — scanning …

Scan complete — 16004.5s elapsed.

  UNI2H TenCrop — Feature File Report
  Root directory : D:\Aamir Gulzar\KSA_project2\dataset\Features\UNI2H_tencrop
  Total .pt files: 1,377,933
  ✓  Valid       : 1,377,933  (100.0 %)
  ✗  Invalid     : 0  (0.0 %)
------------------------------------------------------------------------

  ✓  All files passed validation — no errors detected.
------------------------------------------------------------------------

  TENSOR SHAPE DISTRIBUTION  (valid files only)
------------------------------------------------------------------------
  (10, 1536)                   1377933 files  (100.0%)  ██████████████████████████████
------------------------------------------------------------------------

  AGGREGATE STATISTICS  (across 1,377,933 valid files)
------------------------------------------------------------------------
  Metric                         Mean          Std          Min          Max
  ---------------

In [1]:
import os
from pathlib import Path
from IPython.display import display
import pandas as pd

# ─── CONFIG ───────────────────────────────────────────────────────────────────
ROOT = r"D:\Aamir Gulzar\KSA_project2\dataset\Features"
# ──────────────────────────────────────────────────────────────────────────────

root = Path(ROOT)
model_dirs = sorted([d for d in root.iterdir() if d.is_dir()])

print(f"Found {len(model_dirs)} model folder(s) under:\n  {root}\n")
print("Counting .pt files (this may take a while for large datasets)...\n")

results = []
for model_dir in model_dirs:
    count = sum(1 for _ in model_dir.rglob("*.pt"))
    results.append({"Model": model_dir.name, "PT Files": count, "Status": "✓ Done" if count > 0 else "✗ Empty"})
    print(f"  {model_dir.name:<45} {count:>12,}")

# ── Summary DataFrame ─────────────────────────────────────────────────────────
df = pd.DataFrame(results).sort_values("PT Files", ascending=False).reset_index(drop=True)
df.index += 1  # rank from 1

grand_total = df["PT Files"].sum()
done  = (df["Status"] == "✓ Done").sum()
empty = (df["Status"] == "✗ Empty").sum()

print(f"\n{'='*60}")
print(f"  GRAND TOTAL : {grand_total:,} .pt files")
print(f"  Models Done : {done}  |  Models Empty/Missing : {empty}")
print(f"{'='*60}\n")

display(df.style
    .format({"PT Files": "{:,}"})
    .bar(subset=["PT Files"], color="#4CAF50")
    .set_caption("Model .pt File Counts — sorted by count descending")
)

Found 8 model folder(s) under:
  D:\Aamir Gulzar\KSA_project2\dataset\Features

Counting .pt files (this may take a while for large datasets)...

  CONCH_tencrop                                      153,942
  Conch_v1_5_fivecrop_Updated                      1,377,933
  H_Optimus_1_fivecrop_Updated                     1,377,933
  H_Optimus_1_tencrop                                212,011
  Tencrop_tests                                       12,328
  UNI2H_tencrop                                    1,377,933
  UNI2H_tencrop_test                                   6,164
  Uni_2h_fivecrop_Updated                          1,377,933

  GRAND TOTAL : 5,896,177 .pt files
  Models Done : 8  |  Models Empty/Missing : 0



,Model,PT Files,Status
1,Conch_v1_5_fivecrop_Updated,"1,377,933",✓ Done
2,H_Optimus_1_fivecrop_Updated,"1,377,933",✓ Done
3,UNI2H_tencrop,"1,377,933",✓ Done
4,Uni_2h_fivecrop_Updated,"1,377,933",✓ Done
5,H_Optimus_1_tencrop,"212,011",✓ Done
6,CONCH_tencrop,"153,942",✓ Done
7,Tencrop_tests,"12,328",✓ Done
8,UNI2H_tencrop_test,"6,164",✓ Done


In [2]:
import os
import torch
import random
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd
from IPython.display import display

# ─── CONFIG ───────────────────────────────────────────────────────────────────
FEATURE_DIR   = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Virchow2_fivecrop"
SAMPLE_RATE   = 0.02          # check 2% of files (~26,000 from 1.3M) — change to 1.0 for full check
MAX_WORKERS   = 8             # parallel threads for I/O bound checking
EXPECTED_SHAPE = (5, 2560)    # fivecrop → (5, 2560) | tencrop → (10, 2560)
# ──────────────────────────────────────────────────────────────────────────────

def validate_pt_file(path: Path) -> dict:
    """
    Validates a single .pt file.
    Checks: loadable | correct shape | no NaN | no Inf | non-zero
    """
    result = {
        "file"    : str(path),
        "ok"      : False,
        "error"   : None,
        "shape"   : None,
        "has_nan" : False,
        "has_inf" : False,
        "all_zero": False,
    }
    try:
        tensor = torch.load(path, map_location="cpu", weights_only=True)

        # ── 1. Must be a tensor ───────────────────────────────────────────────
        if not isinstance(tensor, torch.Tensor):
            result["error"] = f"Not a tensor: {type(tensor)}"
            return result

        result["shape"] = tuple(tensor.shape)

        # ── 2. Shape check ────────────────────────────────────────────────────
        if tuple(tensor.shape) != EXPECTED_SHAPE:
            result["error"] = f"Wrong shape: {tuple(tensor.shape)} expected {EXPECTED_SHAPE}"
            return result

        # ── 3. NaN / Inf check ────────────────────────────────────────────────
        result["has_nan"] = bool(torch.isnan(tensor).any())
        result["has_inf"] = bool(torch.isinf(tensor).any())

        if result["has_nan"]:
            result["error"] = "Contains NaN values"
            return result
        if result["has_inf"]:
            result["error"] = "Contains Inf values"
            return result

        # ── 4. All-zero check (likely failed extraction) ──────────────────────
        result["all_zero"] = bool((tensor == 0).all())
        if result["all_zero"]:
            result["error"] = "All-zero tensor (likely corrupt extraction)"
            return result

        result["ok"] = True

    except Exception as e:
        result["error"] = str(e)

    return result


def run_corruption_check(
    feature_dir  : str,
    sample_rate  : float = SAMPLE_RATE,
    max_workers  : int   = MAX_WORKERS,
    expected_shape       = EXPECTED_SHAPE,
):
    root = Path(feature_dir)
    print(f"Scanning for .pt files in:\n  {root}\n")

    # ── Collect all .pt paths ─────────────────────────────────────────────────
    all_files = list(root.rglob("*.pt"))
    total     = len(all_files)
    print(f"Total .pt files found : {total:,}")

    # ── Sample ────────────────────────────────────────────────────────────────
    if sample_rate < 1.0:
        n_sample = max(1, int(total * sample_rate))
        sampled  = random.sample(all_files, n_sample)
        print(f"Sampling {sample_rate*100:.1f}%  →  checking {n_sample:,} files\n")
    else:
        sampled = all_files
        print(f"Checking ALL {total:,} files\n")

    # ── Parallel validation ───────────────────────────────────────────────────
    corrupt  = []
    ok_count = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(validate_pt_file, p): p for p in sampled}
        for future in tqdm(as_completed(futures), total=len(sampled), desc="Validating"):
            res = future.result()
            if res["ok"]:
                ok_count += 1
            else:
                corrupt.append(res)

    # ── Report ────────────────────────────────────────────────────────────────
    n_checked    = len(sampled)
    n_corrupt    = len(corrupt)
    corrupt_rate = n_corrupt / n_checked * 100
    estimated_corrupt = int(corrupt_rate / 100 * total)

    print(f"\n{'='*65}")
    print(f"  Files checked       : {n_checked:,}  ({sample_rate*100:.1f}% sample)")
    print(f"  ✓ Clean             : {ok_count:,}")
    print(f"  ✗ Corrupt           : {n_corrupt:,}  ({corrupt_rate:.2f}%)")
    print(f"  Estimated corrupt   : ~{estimated_corrupt:,} out of {total:,} total")
    print(f"{'='*65}\n")

    if n_corrupt == 0:
        print("✅ No corruption detected in sampled files.")
    else:
        print("⚠️  Corrupt files detected! Breakdown:\n")

        df_corrupt = pd.DataFrame(corrupt)[["file", "shape", "error", "has_nan", "has_inf", "all_zero"]]

        # Error category summary
        error_summary = df_corrupt["error"].value_counts().reset_index()
        error_summary.columns = ["Error Type", "Count"]
        print("── Error Type Breakdown ──────────────────────────────────────")
        display(error_summary)

        print("\n── Sample of Corrupt Files (first 20) ───────────────────────")
        display(df_corrupt.head(20))

        # Save full corrupt list
        out_csv = Path(feature_dir) / "corrupt_files.csv"
        df_corrupt.to_csv(out_csv, index=False)
        print(f"\n💾 Full corrupt file list saved to:\n  {out_csv}")

    return corrupt


# ─── RUN ──────────────────────────────────────────────────────────────────────
corrupt_files = run_corruption_check(
    feature_dir   = FEATURE_DIR,
    sample_rate   = SAMPLE_RATE,       # 0.02 = 2% sample | 1.0 = full check
    max_workers   = MAX_WORKERS,
    expected_shape= EXPECTED_SHAPE,
)
# ```

# ---

# ## Why This is Fast

# The bottleneck with 1.3M files is **I/O**, not compute. The strategy:

# | Technique | Benefit |
# |---|---|
# | **2% random sampling** | Checks ~26K files instead of 1.3M, statistically reliable |
# | **ThreadPoolExecutor** | 8 parallel threads saturate disk I/O simultaneously |
# | **`weights_only=True`** | Fastest possible torch.load, skips unnecessary deserialization |
# | **`map_location="cpu"`** | No GPU overhead for validation |

# ---

# ## What it Checks Per File
# ```
# 1. Loadable         → file not truncated/empty
# 2. Is a Tensor      → not a dict or list accidentally saved
# 3. Shape == (5,2560)→ extraction completed fully
# 4. No NaN           → forward pass didn't explode
# 5. No Inf           → no overflow during autocast
# 6. Not all-zero     → extraction didn't silently fail

Scanning for .pt files in:
  D:\Aamir Gulzar\KSA_project2\dataset\Features\Virchow2_fivecrop

Total .pt files found : 1,377,933
Sampling 2.0%  →  checking 27,558 files



Validating: 100%|██████████| 27558/27558 [09:03<00:00, 50.72it/s] 



  Files checked       : 27,558  (2.0% sample)
  ✓ Clean             : 27,558
  ✗ Corrupt           : 0  (0.00%)
  Estimated corrupt   : ~0 out of 1,377,933 total

✅ No corruption detected in sampled files.


In [1]:
import os
import torch
import random
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd
from IPython.display import display

# ─── CONFIG ───────────────────────────────────────────────────────────────────
FEATURE_DIR    = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Virchow2_fivecrop"
SAMPLE_RATE    = 0.02
MAX_WORKERS    = 8
EXPECTED_SHAPE = (5, 2560)
# ──────────────────────────────────────────────────────────────────────────────

def validate_pt_file(path: Path) -> dict:
    result = {
        "file"      : str(path),
        "ok"        : False,
        "error"     : None,
        "shape"     : None,
        "dtype"     : None,
        "min"       : None,
        "max"       : None,
        "mean"      : None,
        "std"       : None,
        "has_nan"   : False,
        "has_inf"   : False,
        "all_zero"  : False,
        "file_size_kb": round(path.stat().st_size / 1024, 2),
    }
    try:
        tensor = torch.load(path, map_location="cpu", weights_only=True)

        if not isinstance(tensor, torch.Tensor):
            result["error"] = f"Not a tensor: {type(tensor)}"
            return result

        result["shape"] = tuple(tensor.shape)
        result["dtype"] = str(tensor.dtype)

        # ── Value stats ───────────────────────────────────────────────────────
        t_float = tensor.float()
        result["min"]  = round(float(t_float.min()), 6)
        result["max"]  = round(float(t_float.max()), 6)
        result["mean"] = round(float(t_float.mean()), 6)
        result["std"]  = round(float(t_float.std()), 6)

        # ── Checks ────────────────────────────────────────────────────────────
        if tuple(tensor.shape) != EXPECTED_SHAPE:
            result["error"] = f"Wrong shape: {tuple(tensor.shape)} expected {EXPECTED_SHAPE}"
            return result

        result["has_nan"] = bool(torch.isnan(t_float).any())
        result["has_inf"] = bool(torch.isinf(t_float).any())

        if result["has_nan"]:
            result["error"] = "Contains NaN values"
            return result
        if result["has_inf"]:
            result["error"] = "Contains Inf values"
            return result

        result["all_zero"] = bool((t_float == 0).all())
        if result["all_zero"]:
            result["error"] = "All-zero tensor"
            return result

        result["ok"] = True

    except Exception as e:
        result["error"] = str(e)

    return result


def run_corruption_check(
    feature_dir   : str,
    sample_rate   : float = SAMPLE_RATE,
    max_workers   : int   = MAX_WORKERS,
    expected_shape        = EXPECTED_SHAPE,
):
    root = Path(feature_dir)
    print(f"Scanning for .pt files in:\n  {root}\n")

    all_files = list(root.rglob("*.pt"))
    total     = len(all_files)
    print(f"Total .pt files found : {total:,}")

    if sample_rate < 1.0:
        n_sample = max(1, int(total * sample_rate))
        sampled  = random.sample(all_files, n_sample)
        print(f"Sampling {sample_rate*100:.1f}%  →  checking {n_sample:,} files\n")
    else:
        sampled = all_files
        print(f"Checking ALL {total:,} files\n")

    # ── Parallel validation ───────────────────────────────────────────────────
    results  = []
    corrupt  = []
    ok_count = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(validate_pt_file, p): p for p in sampled}
        for future in tqdm(as_completed(futures), total=len(sampled), desc="Validating"):
            res = future.result()
            results.append(res)
            if res["ok"]:
                ok_count += 1
            else:
                corrupt.append(res)

    df_all = pd.DataFrame(results)

    # ─────────────────────────────────────────────────────────────────────────
    # REPORT
    # ─────────────────────────────────────────────────────────────────────────
    n_checked      = len(sampled)
    n_corrupt      = len(corrupt)
    corrupt_rate   = n_corrupt / n_checked * 100
    est_corrupt    = int(corrupt_rate / 100 * total)

    print(f"\n{'═'*65}")
    print(f"  VALIDATION SUMMARY")
    print(f"{'═'*65}")
    print(f"  Total files in dir  : {total:,}")
    print(f"  Files checked       : {n_checked:,}  ({sample_rate*100:.1f}% sample)")
    print(f"  ✓ Clean             : {ok_count:,}")
    print(f"  ✗ Corrupt           : {n_corrupt:,}  ({corrupt_rate:.2f}%)")
    print(f"  Estimated corrupt   : ~{est_corrupt:,} out of {total:,} total")
    print(f"{'═'*65}\n")

    # ── Shape distribution ────────────────────────────────────────────────────
    print(f"{'─'*65}")
    print(f"  SHAPE DISTRIBUTION  (expected: {expected_shape})")
    print(f"{'─'*65}")
    shape_counts = df_all["shape"].value_counts().reset_index()
    shape_counts.columns = ["Shape", "Count"]
    shape_counts["% of checked"] = (shape_counts["Count"] / n_checked * 100).round(2)
    shape_counts["Match Expected"] = shape_counts["Shape"].apply(
        lambda s: "✓" if str(s) == str(expected_shape) else "✗"
    )
    display(shape_counts)

    # ── Dtype distribution ────────────────────────────────────────────────────
    print(f"\n{'─'*65}")
    print(f"  DTYPE DISTRIBUTION")
    print(f"{'─'*65}")
    dtype_counts = df_all["dtype"].value_counts().reset_index()
    dtype_counts.columns = ["DType", "Count"]
    display(dtype_counts)

    # ── File size stats ───────────────────────────────────────────────────────
    print(f"\n{'─'*65}")
    print(f"  FILE SIZE STATS (KB)")
    print(f"{'─'*65}")
    size_stats = df_all["file_size_kb"].describe().round(2).to_frame().T
    display(size_stats)

    # ── Value stats across all clean files ───────────────────────────────────
    df_clean = df_all[df_all["ok"] == True]
    if len(df_clean) > 0:
        print(f"\n{'─'*65}")
        print(f"  TENSOR VALUE STATS  (across {len(df_clean):,} clean files)")
        print(f"{'─'*65}")
        value_stats = df_clean[["min", "max", "mean", "std"]].describe().round(6)
        display(value_stats)

        # Per-crop stats (mean across files, per crop row)
        print(f"\n{'─'*65}")
        print(f"  PER-CROP MEAN STATS  (mean value per crop position)")
        print(f"{'─'*65}")
        print("  (Loading 50 random clean files to compute per-crop breakdown...)")
        sample_clean = df_clean.sample(min(50, len(df_clean)))
        crop_means = []
        for _, row in sample_clean.iterrows():
            try:
                t = torch.load(row["file"], map_location="cpu", weights_only=True).float()
                crop_means.append(t.mean(dim=1).numpy())   # (5,) mean per crop
            except:
                pass
        if crop_means:
            crop_arr = np.stack(crop_means)   # (N, 5)
            df_crop = pd.DataFrame(
                crop_arr.mean(axis=0).reshape(1, -1),
                columns=[f"Crop {i} mean" for i in range(crop_arr.shape[1])]
            ).round(6)
            display(df_crop)

    # ── Corrupt file breakdown ────────────────────────────────────────────────
    if n_corrupt > 0:
        print(f"\n{'─'*65}")
        print(f"  ✗ CORRUPT FILE BREAKDOWN")
        print(f"{'─'*65}")
        df_corrupt = pd.DataFrame(corrupt)[["file", "shape", "dtype", "error",
                                            "has_nan", "has_inf", "all_zero",
                                            "file_size_kb"]]
        error_summary = df_corrupt["error"].value_counts().reset_index()
        error_summary.columns = ["Error Type", "Count"]
        print("\n  Error Type Summary:")
        display(error_summary)
        print("\n  First 20 corrupt files:")
        display(df_corrupt.head(20))

        out_csv = Path(feature_dir) / "corrupt_files.csv"
        df_corrupt.to_csv(out_csv, index=False)
        print(f"\n  💾 Full corrupt list saved → {out_csv}")
    else:
        print("✅ No corruption detected in sampled files.")

    return df_all, corrupt


# ─── RUN ──────────────────────────────────────────────────────────────────────
df_results, corrupt_files = run_corruption_check(
    feature_dir    = FEATURE_DIR,
    sample_rate    = SAMPLE_RATE,
    max_workers    = MAX_WORKERS,
    expected_shape = EXPECTED_SHAPE,
)

Scanning for .pt files in:
  D:\Aamir Gulzar\KSA_project2\dataset\Features\Virchow2_fivecrop

Total .pt files found : 1,377,933
Sampling 2.0%  →  checking 27,558 files



Validating: 100%|██████████| 27558/27558 [10:07<00:00, 45.37it/s]


═════════════════════════════════════════════════════════════════
  VALIDATION SUMMARY
═════════════════════════════════════════════════════════════════
  Total files in dir  : 1,377,933
  Files checked       : 27,558  (2.0% sample)
  ✓ Clean             : 27,558
  ✗ Corrupt           : 0  (0.00%)
  Estimated corrupt   : ~0 out of 1,377,933 total
═════════════════════════════════════════════════════════════════

─────────────────────────────────────────────────────────────────
  SHAPE DISTRIBUTION  (expected: (5, 2560))
─────────────────────────────────────────────────────────────────


,Shape,Count,% of checked,Match Expected
0,"(5, 2560)",27558,100.0,✓



─────────────────────────────────────────────────────────────────
  DTYPE DISTRIBUTION
─────────────────────────────────────────────────────────────────


...


─────────────────────────────────────────────────────────────────
  TENSOR VALUE STATS  (across 27,558 clean files)
─────────────────────────────────────────────────────────────────


,min,max,mean,std
count,27558.000000,27558.000000,27558.000000,27558.000000
mean,-26.822945,8.939765,-0.013337,0.952878
std,9.302392,3.510996,0.009172,0.094769
min,-47.288052,3.482754,-0.040039,0.644961
25%,-34.345255,6.932152,-0.020522,0.884214
50%,-28.272894,7.935870,-0.011701,0.947038
75%,-19.967798,9.259134,-0.006793,1.023067
max,-3.210928,28.974716,0.010153,1.225032



─────────────────────────────────────────────────────────────────
  PER-CROP MEAN STATS  (mean value per crop position)
─────────────────────────────────────────────────────────────────
  (Loading 50 random clean files to compute per-crop breakdown...)


,Crop 0 mean,Crop 1 mean,Crop 2 mean,Crop 3 mean,Crop 4 mean
0,-0.013787,-0.013345,-0.014445,-0.013243,-0.014446


✅ No corruption detected in sampled files.
